# State of Data Brazil — Pipeline de Dados (Camada Silver, PySpark)

**Tech Challenge — Unificação e limpeza**

Versão em PySpark da unificação Silver — parte da Bronze (já com nomes de coluna
saneados) e usa o dicionário de dados (`data/dictionary/mapa_campos.csv`) para
selecionar só os **80 campos marcados como "Sim"** na planilha de mapeamento e
renomeá-los para nomes padronizados, unificando os schemas divergentes de
2023/2024/2025 num único DataFrame.

O dicionário já foi validado linha a linha contra as colunas reais dos 3 CSVs
(inclusive corrigindo 6 inconsistências que existiam na planilha original — ver
notebook anterior e o histórico da conversa) e suas colunas `coluna_2023`,
`coluna_2024`, `coluna_2025` já apontam para os nomes **saneados** que a Bronze
gerou (não os nomes brutos).


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve() / 'src'))
from pipeline_utils import find_project_root, get_spark_session

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

PROJECT_ROOT = find_project_root()
BRONZE_DIR = PROJECT_ROOT / 'data' / 'bronze'
SILVER_DIR = PROJECT_ROOT / 'data' / 'silver'
DICT_PATH = PROJECT_ROOT / 'data' / 'dictionary' / 'mapa_campos.csv'
SILVER_DIR.mkdir(parents=True, exist_ok=True)

spark = get_spark_session('state-of-data-silver')
print("Raiz do projeto:", PROJECT_ROOT)


c:\Users\fhca02\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Raiz do projeto: c:\Users\fhca02\OneDrive - Church of Jesus Christ\Documents\Fabricio\Pos FIAP\Fase 3\Tech Challenge - Fase 3\state-of-data-pipeline_1


## 1. Carregamento da Bronze e do dicionário de dados

In [2]:
dfs_bronze = {}
for ano in (2023, 2024, 2025):
    dfs_bronze[ano] = spark.read.parquet(str(BRONZE_DIR / f'ano={ano}'))
    print(f"{ano}: {dfs_bronze[ano].count():,} linhas x {len(dfs_bronze[ano].columns)} colunas".replace(',', '.'))


2023: 5.293 linhas x 402 colunas
2024: 5.217 linhas x 406 colunas
2025: 3.495 linhas x 391 colunas


In [3]:
# o dicionario e uma tabela de referencia pequena (80 linhas) -- ler com pandas
# aqui e so uma conveniencia local; num Glue Job o mesmo csv pode vir do S3
mapa_campos = pd.read_csv(DICT_PATH)
print(f"{len(mapa_campos)} campos no dicionario de dados")
mapa_campos[['campo_padronizado', 'coluna_2023', 'coluna_2024', 'coluna_2025']].head(8)


80 campos no dicionario de dados


,campo_padronizado,coluna_2023,coluna_2024,coluna_2025
0,ai_generativa_e_llm_e_uma_prioridade,p3_e_ai_generativa_e_uma_prioridade_em_sua_emp...,c_3_e_ai_generativa_e_llm_e_uma_prioridade,c_3_e_ai_generativa_e_llm_e_uma_prioridade
1,area_de_formacao,p1_m_area_de_formacao,c_1_m_area_de_formacao,c_1_m_area_de_formacao
2,aspectos_prejudicados,p1_f_aspectos_prejudicados,c_1_f_aspectos_prejudicados,c_1_f_aspectos_prejudicados
3,atitude_em_caso_de_retorno_presencial,p2_t_caso_sua_empresa_decida_pelo_modelo_100_p...,c_2_t_atitude_em_caso_de_retorno_presencial,c_2_s_atitude_em_caso_de_retorno_presencial
4,atua_como_gestor,p2_d_gestor,c_2_d_atua_como_gestor,c_2_d_atua_como_gestor
5,banco_de_dados_dia_a_dia,p4_g_quais_dos_bancos_de_dados_fontes_de_dados...,c_4_g_banco_de_dados_dia_a_dia,c_4_d_banco_de_dados_dia_a_dia
6,cargo_atual,p2_f_cargo_atual,c_2_f_cargo_atual,c_2_f_cargo_atual
7,cargo_como_gestor,p2_e_cargo_como_gestor,c_2_e_cargo_como_gestor,c_2_e_cargo_como_gestor


## 2. Validação do dicionário contra a Bronze

Confere que toda coluna saneada citada no dicionário existe de fato no ano correspondente da Bronze.

In [4]:
problemas = []
for ano, col_dicionario in [(2023, 'coluna_2023'), (2024, 'coluna_2024'), (2025, 'coluna_2025')]:
    colunas_reais = set(dfs_bronze[ano].columns)
    for _, row in mapa_campos.dropna(subset=[col_dicionario]).iterrows():
        if row[col_dicionario] not in colunas_reais:
            problemas.append((ano, row['campo_padronizado'], row[col_dicionario]))

if problemas:
    print(f"ATENCAO: {len(problemas)} colunas do dicionario nao encontradas na Bronze:")
    for p in problemas:
        print(' ', p)
else:
    print("OK: todas as colunas citadas no dicionario existem na camada Bronze correspondente.")


OK: todas as colunas citadas no dicionario existem na camada Bronze correspondente.


## 3. Seleção e renomeação por ano

In [5]:
def selecionar_e_renomear(df_bronze, mapa, coluna_ano):
    mapa_ano = mapa.dropna(subset=[coluna_ano])
    colunas_lineage = ['_ano_pesquisa', '_arquivo_origem', '_data_ingestao']
    exprs = [F.col(c).alias(c) for c in colunas_lineage]
    exprs += [F.col(row[coluna_ano]).alias(row['campo_padronizado']) for _, row in mapa_ano.iterrows()]
    return df_bronze.select(*exprs)

df_2023_silver = selecionar_e_renomear(dfs_bronze[2023], mapa_campos, 'coluna_2023')
df_2024_silver = selecionar_e_renomear(dfs_bronze[2024], mapa_campos, 'coluna_2024')
df_2025_silver = selecionar_e_renomear(dfs_bronze[2025], mapa_campos, 'coluna_2025')

for ano, df in [(2023, df_2023_silver), (2024, df_2024_silver), (2025, df_2025_silver)]:
    print(f"{ano}: {df.count():,} linhas x {len(df.columns)} colunas".replace(',', '.'))


2023: 5.293 linhas x 78 colunas
2024: 5.217 linhas x 82 colunas
2025: 3.495 linhas x 80 colunas


## 4. União dos três anos em um schema único

`unionByName(..., allowMissingColumns=True)` alinha as colunas pelo nome — campos que não existem em um ano viram nulos nas linhas daquele ano.

In [6]:
df_silver = (
    df_2023_silver
    .unionByName(df_2024_silver, allowMissingColumns=True)
    .unionByName(df_2025_silver, allowMissingColumns=True)
)
print(f"Silver unificado: {df_silver.count():,} linhas x {len(df_silver.columns)} colunas".replace(',', '.'))
df_silver.limit(3).toPandas()


Silver unificado: 14.005 linhas x 83 colunas


c:\Users\fhca02\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,_ano_pesquisa,_arquivo_origem,_data_ingestao,ai_generativa_e_llm_e_uma_prioridade,area_de_formacao,aspectos_prejudicados,atitude_em_caso_de_retorno_presencial,atua_como_gestor,banco_de_dados_dia_a_dia,cargo_atual,...,token,uf_onde_mora,usa_chatgpt_ou_copilot_no_trabalho,vive_no_brasil,vive_no_estado_de_formacao,data_hora_envio,pais_onde_mora,regiao_de_origem,uf_de_origem,empresa_esta_conseguindo_ter_bons_resultados_com_llms
0,2023,2023.csv,2026-09-02T04:29:26.495888+00:00,NaN,Computação / Engenharia de Software / Sistemas...,NaN,Vou procurar outra oportunidade no modelo 100%...,0,"SQLite, SQL SERVER, S3, Redis, Hive",Cientista de Dados/Data Scientist,...,001b2d1qtli8t9z7oqgdhj001b2d4i0g,MG,Não utilizo nenhum tipo de solução de IA Gener...,1,1,NaN,NaN,NaN,NaN,NaN
1,2023,2023.csv,2026-09-02T04:29:26.495888+00:00,NaN,Computação / Engenharia de Software / Sistemas...,NaN,Vou procurar outra oportunidade no modelo híbr...,0,S3,Analista de BI/BI Analyst,...,0026aa3fwd78u0026asg7456tfkjg2cs,ES,Utilizo soluções pagas de AI Generativa (como ...,1,1,NaN,NaN,NaN,NaN,NaN
2,2023,2023.csv,2026-09-02T04:29:26.495888+00:00,NaN,Computação / Engenharia de Software / Sistemas...,Atenção dada pelas pessoas diante das minhas o...,Vou aceitar e retornar ao modelo 100% presencial,0,"S3, Amazon Athena, Hive",Analista de Dados/Data Analyst,...,00r21rb9pusd1b0v7ew00r21rw3dy69w,SP,Utilizo apenas soluções gratuitas (como por ex...,1,1,NaN,NaN,NaN,NaN,NaN


## 5. Tipagem básica

In [7]:
df_silver = (
    df_silver
    .withColumn('data_hora_envio', F.to_timestamp('data_hora_envio', 'dd/MM/yyyy HH:mm:ss'))
    .withColumn('idade', F.col('idade').cast(IntegerType()))
    .withColumn('_ano_pesquisa', F.col('_ano_pesquisa').cast(IntegerType()))
)

df_silver.select('token', 'data_hora_envio', 'idade', '_ano_pesquisa').show(5, truncate=False)


+--------------------------------+---------------+-----+-------------+
|token                           |data_hora_envio|idade|_ano_pesquisa|
+--------------------------------+---------------+-----+-------------+
|001b2d1qtli8t9z7oqgdhj001b2d4i0g|NULL           |31   |2023         |
|0026aa3fwd78u0026asg7456tfkjg2cs|NULL           |30   |2023         |
|00r21rb9pusd1b0v7ew00r21rw3dy69w|NULL           |37   |2023         |
|00urm3jf2cek12w6ygue00urm3jzd17j|NULL           |22   |2023         |
|00v0az4g792svil00vn6y1kfm9hq8vy9|NULL           |34   |2023         |
+--------------------------------+---------------+-----+-------------+
only showing top 5 rows


## 6. Padronização de valores categóricos

Quatro inconsistências de **codificação** (não de schema) apareceram entre as
edições da pesquisa e precisam ser resolvidas na Silver, antes da Gold — senão
qualquer agregação por essas colunas conta a mesma categoria como se fossem
duas. Todas identificadas comparando `value_counts()` por campo entre os 3 anos:

1. **Campos booleanos com codificação mista.** Seis campos que deveriam ser
   Sim/Não vêm com valores `'1'`/`'0'` em 2023 e 2025, mas `'TRUE'`/`'FALSE'` em
   2024 (exportação do formulário mudou de formato entre edições):
   `atua_como_gestor`, `possui_data_lake`, `possui_data_warehouse`,
   `satisfeito_atualmente`, `vive_no_brasil`, `vive_no_estado_de_formacao`.
   Padronizados para `'Sim'` / `'Não'`, no mesmo estilo textual já usado pelos
   outros campos binários da base (ex: `pcd`).
2. **`cargo_atual` com um rótulo duplicado entre edições.** A edição 2025 passou a
   incluir "Arquiteto de Dados" explicitamente no rótulo de Engenheiro de Dados
   (`"Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect"`),
   enquanto 2023/2024 usam `"Engenheiro de Dados/Data Engineer/Data Architect"`
   para a mesma função. Consolidado no rótulo mais curto.
3. **`ai_generativa_e_llm_e_uma_prioridade` com a mesma opção reescrita entre
   edições** (`"...iniciativas isoladas e pouco foco)"` vs
   `"...tratam-se de iniciativas isoladas e com pouco foco)"`). Consolidado no
   texto mais recente.
4. **`numero_de_funcionarios` com um valor corrompido** (`"de 501 a 100"`,
   2 ocorrências) — claramente uma versão truncada de `"de 501 a 1.000"` (as
   demais faixas seguem o padrão `de X a Y`). Corrigido.
5. **`faixa_salarial` com dois valores corrompidos** (1 ocorrência cada):
   `"de R$ 101/mês a R$ 2.000/mês"` → `"de R$ 1.001/mês a R$ 2.000/mês"` e
   `"de R$ 25.001/mês a R$ 3000/mês"` → `"de R$ 25.001/mês a R$ 30.000/mês"`
   (ambos batem exatamente com faixas já existentes na base, faltando dígitos).

Um caso parecido em `tempo_de_experiencia_em_dados` (`"de 4 a 6 anos"` só
aparece em 2023, `"de 5 a 6 anos"` nos 3 anos) **não** foi tratado como erro —
é uma mudança real de faixa entre edições da pesquisa, não uma corrupção de
texto, então as duas faixas foram mantidas separadas.


In [8]:
# 1) campos booleanos com codificacao mista (1/0 em 2023+2025, TRUE/FALSE em 2024)
campos_booleanos = [
    'atua_como_gestor', 'possui_data_lake', 'possui_data_warehouse',
    'satisfeito_atualmente', 'vive_no_brasil', 'vive_no_estado_de_formacao',
]

mapa_bool = {'1': 'Sim', 'TRUE': 'Sim', '0': 'Não', 'FALSE': 'Não'}
bool_expr = F.create_map([F.lit(x) for pair in mapa_bool.items() for x in pair])

for c in campos_booleanos:
    df_silver = df_silver.withColumn(c, bool_expr[F.col(c)])

# 2) cargo_atual: consolida o rotulo de 2025 no rotulo usado em 2023/2024
# 3) ai_generativa_e_llm_e_uma_prioridade: consolida reescrita da mesma opcao
# 4) numero_de_funcionarios: corrige valor truncado/corrompido
mapa_consolidacao = {
    'cargo_atual': {
        'Engenheiro de Dados/Arquiteto de Dados/Data Engineer/Data Architect':
            'Engenheiro de Dados/Data Engineer/Data Architect',
    },
    'ai_generativa_e_llm_e_uma_prioridade': {
        'Mais ou menos... É uma das várias iniciativas que estamos impulsionando, mas não é uma prioridade (iniciativas isoladas e pouco foco).':
            'Mais ou menos... É uma das várias iniciativas que estamos impulsionando, mas não é uma prioridade (tratam-se de iniciativas isoladas e com pouco foco).',
    },
    'numero_de_funcionarios': {
        'de 501 a 100': 'de 501 a 1.000',
    },
    'faixa_salarial': {
        'de R$ 101/mês a R$ 2.000/mês': 'de R$ 1.001/mês a R$ 2.000/mês',
        'de R$ 25.001/mês a R$ 3000/mês': 'de R$ 25.001/mês a R$ 30.000/mês',
    },
}

for coluna, mapa in mapa_consolidacao.items():
    expr = F.col(coluna)
    for de, para in mapa.items():
        expr = F.when(F.col(coluna) == de, F.lit(para)).otherwise(expr)
    df_silver = df_silver.withColumn(coluna, expr)

print("Verificacao pos-padronizacao:")
for c in campos_booleanos:
    df_silver.groupBy(c).count().orderBy(F.desc('count')).show(truncate=False)
df_silver.groupBy('cargo_atual').count().orderBy(F.desc('count')).show(20, truncate=False)
df_silver.groupBy('ai_generativa_e_llm_e_uma_prioridade').count().orderBy(F.desc('count')).show(10, truncate=False)
df_silver.groupBy('numero_de_funcionarios').count().orderBy(F.desc('count')).show(10, truncate=False)
df_silver.groupBy('faixa_salarial').count().orderBy(F.desc('count')).show(20, truncate=False)


Verificacao pos-padronizacao:
+----------------+-----+
|atua_como_gestor|count|
+----------------+-----+
|Não             |10176|
|Sim             |2668 |
|NULL            |1161 |
+----------------+-----+

+----------------+-----+
|possui_data_lake|count|
+----------------+-----+
|NULL            |11608|
|Sim             |1976 |
|Não             |421  |
+----------------+-----+

+---------------------+-----+
|possui_data_warehouse|count|
+---------------------+-----+
|NULL                 |11633|
|Sim                  |1949 |
|Não                  |423  |
+---------------------+-----+

+---------------------+-----+
|satisfeito_atualmente|count|
+---------------------+-----+
|Sim                  |8982 |
|Não                  |3862 |
|NULL                 |1161 |
+---------------------+-----+

+--------------+-----+
|vive_no_brasil|count|
+--------------+-----+
|Sim           |13623|
|Não           |382  |
+--------------+-----+

+--------------------------+-----+
|vive_no_estado_de_for

## 7. Checagem de qualidade

In [9]:
total = df_silver.count()
nulos = df_silver.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total * 100).alias(c) for c in df_silver.columns
])
nulos_pd = nulos.toPandas().T.rename(columns={0: 'pct_nulos'}).sort_values('pct_nulos', ascending=False)
print("Top 15 campos com mais nulos (%):")
print(nulos_pd.head(15).round(1))


c:\Users\fhca02\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Top 15 campos com mais nulos (%):
                                                    pct_nulos
pais_onde_mora                                           98.1
empresa_esta_conseguindo_ter_bons_resultados_co...       95.4
oportunidade_buscada                                     94.6
experiencia_em_processos_seletivos                       94.6
tempo_em_busca_de_oportunidade                           94.5
objetivo_na_area_de_dados                                89.4
uf_de_origem                                             88.3
regiao_de_origem                                         88.3
ferramentas_de_qualidade_de_dados_dia_a_dia              88.2
tecnologia_data_warehouse                                86.5
tecnologia_data_lake                                     86.5
tecnicas_e_metodos_ds                                    85.6
rotina_como_ds                                           85.6
tecnologias_ds                                           85.6
maior_tempo_gasto_como_ds           

In [10]:
dup = (
    df_silver.groupBy('_ano_pesquisa', 'token').count()
    .filter(F.col('count') > 1)
    .count()
)
print(f"Tokens duplicados dentro do mesmo ano: {dup}")

print("\nLinhas por ano:")
df_silver.groupBy('_ano_pesquisa').count().orderBy('_ano_pesquisa').show()


Tokens duplicados dentro do mesmo ano: 3

Linhas por ano:
+-------------+-----+
|_ano_pesquisa|count|
+-------------+-----+
|         2023| 5293|
|         2024| 5217|
|         2025| 3495|
+-------------+-----+



## 8. Persistência da camada Silver

In [11]:
out_path = SILVER_DIR / 'pesquisa_unificada'
df_silver.coalesce(1).write.mode('overwrite').parquet(str(out_path))
print(f"Salvo: {out_path}")


Salvo: c:\Users\fhca02\OneDrive - Church of Jesus Christ\Documents\Fabricio\Pos FIAP\Fase 3\Tech Challenge - Fase 3\state-of-data-pipeline_1\data\silver\pesquisa_unificada


In [12]:
df_check = spark.read.parquet(str(out_path))
status = 'OK' if (df_check.count(), len(df_check.columns)) == (df_silver.count(), len(df_silver.columns)) else 'DIVERGENTE'
print(f"Shape lido: ({df_check.count()}, {len(df_check.columns)}) | esperado: ({df_silver.count()}, {len(df_silver.columns)}) | {status}")


Shape lido: (14005, 83) | esperado: (14005, 83) | OK


## 9. Próximos passos

- **Gold:** construir as tabelas agregadas (`gold_respondentes` + os 7 marts
  específicos) que respondem às perguntas de negócio do desafio.
- **AWS:** este código roda como Glue Job trocando `get_spark_session` local por
  `GlueContext` e os paths locais por `s3://<bucket>/...`.
